# Day 092 Project — Strategy Lab

Run all four strategies plus buy-and-hold on 252 days of synthetic OHLCV data. Print the comparison table and identify which strategy has the highest Sharpe ratio.

In [ ]:
import pandas as pd, math, warnings

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })

def _sma(s, w):  return s.rolling(w).mean()
def _ema(s, w):  return s.ewm(span=w, adjust=False).mean()
def _rsi(s, w):
    d = s.diff()
    g = d.clip(lower=0).rolling(w).mean()
    l = (-d.clip(upper=0)).rolling(w).mean()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rs = g / l
    return 100 - (100 / (1 + rs))
def sma_crossover(df, fast=20, slow=50):
    close = df["Close"]
    return (_sma(close, fast) > _sma(close, slow)).fillna(False).astype(int)
def rsi_mean_reversion(df, window=14, oversold=30, overbought=70):
    rsi_s  = _rsi(df["Close"], window)
    signal = pd.Series(float("nan"), index=df.index)
    signal[rsi_s < oversold]   = 1.0
    signal[rsi_s > overbought] = 0.0
    return signal.ffill().fillna(0).astype(int)
def macd_cross(df, fast=12, slow=26, signal=9):
    close       = df["Close"]
    macd_line   = _ema(close, fast) - _ema(close, slow)
    signal_line = _ema(macd_line, signal)
    return (macd_line > signal_line).astype(int)
def combined_signal(df, fast=20, slow=50, macd_fast=12, macd_slow=26, macd_sig=9):
    sma_sig  = sma_crossover(df, fast, slow)
    macd_sig = macd_cross(df, macd_fast, macd_slow, macd_sig)
    return ((sma_sig == 1) & (macd_sig == 1)).astype(int)
def _compute_returns(df):  return df["Close"].pct_change()
def _compute_equity(r):    return (1 + r.fillna(0)).cumprod()
def _max_dd(eq):
    peak = eq.cummax()
    return float(((eq - peak) / peak).min())
def _sharpe(r):
    c = r.dropna()
    if len(c) == 0 or c.std() == 0: return 0.0
    return float(c.mean() / c.std() * (252 ** 0.5))
def run_backtest(df, signals, label="strategy"):
    mr  = _compute_returns(df)
    pos = signals.shift(1).fillna(0)
    sr  = pos * mr
    eq  = _compute_equity(sr)
    c   = sr.dropna(); n = len(c)
    tr  = float(eq.iloc[-1] - 1.0)
    base = 1.0 + tr
    ar  = float(base ** (252.0 / max(n, 1)) - 1) if base > 0 else -1.0
    pos_diff = pos.diff().fillna(0)
    return {
        "label":             label,
        "total_return":      tr,
        "annualized_return": ar,
        "sharpe_ratio":      _sharpe(sr),
        "max_drawdown":      _max_dd(eq),
        "win_rate":          float((c > 0).sum() / max(n, 1)),
        "n_trades":          int((pos_diff != 0).sum()),
        "equity":            eq,
    }


In [ ]:
df = _synthetic(n=252)

strategies = [
    ("Buy-Hold", pd.Series(1, index=df.index)),
    ("SMA-cross", sma_crossover(df)),
    ("RSI-MR",    rsi_mean_reversion(df, oversold=35, overbought=65)),
    ("MACD",      macd_cross(df)),
    ("Combined",  combined_signal(df)),
]
results = [run_backtest(df, sig, label) for label, sig in strategies]


In [ ]:
labels = [r["label"] for r in results]
print(f"{'Metric':<22}", " ".join(f"{l:>10}" for l in labels))
print("-" * (22 + 11 * len(labels)))
for key, fmt in [
    ("total_return",      ".2%"),
    ("annualized_return", ".2%"),
    ("sharpe_ratio",      ".3f"),
    ("max_drawdown",      ".2%"),
    ("win_rate",          ".2%"),
    ("n_trades",          "d"),
]:
    vals = []
    for r in results:
        v = r[key]
        vals.append(f"{v:>{10}{fmt}}" if fmt != "d" else f"{v:>10d}")
    print(f"{key:<22}", " ".join(vals))

best = max(results, key=lambda r: r["sharpe_ratio"])
print(f"\nBest Sharpe: {best['label']} ({best['sharpe_ratio']:.3f})")
